# Extracting data

### Importing Required Libraries

In [1]:
import pandas as pd
import requests
import json

### Reading MIBOR-OIS rates

In [6]:
def get_MIBOR_OIS(url):
  r1 = requests.get(url)

  r1.raise_for_status()

  data1 = r1.json()

  mibor_ois_df = pd.DataFrame(data1)

  mibor_ois_df = pd.concat(
      [mibor_ois_df.drop(columns="benchMarkTrendData"),
      mibor_ois_df["benchMarkTrendData"].apply(pd.Series)],
      axis=1
  )

  mibor_ois_df = pd.concat(
      [mibor_ois_df.drop(columns="benchMarkTrend"),
      mibor_ois_df['benchMarkTrend'].apply(pd.Series)],
      axis= 1
      )

  #flatten rates data for each date
  for i in range(61):
    mibor_ois_df = pd.concat(
        [mibor_ois_df.drop(columns= i),
        mibor_ois_df[i].apply(pd.Series)],
        axis= 1
          )

  #stacks rate data of different dates into single column
  records = []

  for i in range(1, mibor_ois_df.shape[1], 2):
    temp = pd.DataFrame({
        "productName": mibor_ois_df.iloc[:, 0],
        "publishDate": mibor_ois_df.iloc[:, i],
        "rate": mibor_ois_df.iloc[:, i + 1]
    })

    records.append(temp)

  result = pd.concat(records, ignore_index=True)

  result["publishDate"] = pd.to_datetime(result["publishDate"])

  #pivot the dataframe into datewise data
  final_mibor_ois_df = result.pivot(
      index="publishDate",
      columns="productName",
      values="rate"
  ).sort_index(ascending= False)

  return final_mibor_ois_df

In [8]:
mibor_ois_url = "https://www.fbil.org.in/wasdm/miborois/fetchcommongraph/3M"

mibor_ois_df = get_MIBOR_OIS(mibor_ois_url)

mibor_ois_df.head()

productName,MIBOR-OIS - 1Y,MIBOR-OIS - 5Y,MIBOR-OIS - 6M
publishDate,,,
2026-07-03,5.78,6.19,5.48
2026-07-02,5.78,6.19,5.48
2026-07-01,5.77,6.20,5.47
2026-06-30,5.75,6.17,5.45
2026-06-29,5.78,6.19,5.48


### Reading overnight OIS rates

In [3]:
def get_ON_OIS(url):
  r = requests.get(url)
  r.raise_for_status()

  # print(r.status_code)

  data = r.json()

  on_MIBOR_df = pd.DataFrame(data)

  on_MIBOR_df = pd.concat(
      [on_MIBOR_df.drop(columns="benchMarkTrend"),
      on_MIBOR_df["benchMarkTrend"].apply(pd.Series)],
      axis=1
  )

  on_MIBOR_df = on_MIBOR_df.rename(columns = {'publishDate': 'Date'})

  on_MIBOR_df['Date'] = pd.to_datetime(on_MIBOR_df['Date'])
  on_MIBOR_df.drop(columns= ['productName'], inplace= True)

  return on_MIBOR_df

In [11]:
url_on_OIS = "https://www.fbil.org.in/wasdm/ovnmibor/fetchgraphdata/3M"

on_MIBOR_df = get_ON_OIS(url_on_OIS)

on_MIBOR_df.tail()

,Date,rate
56,2026-04-10,5.09
57,2026-04-09,5.13
58,2026-04-08,5.13
59,2026-04-07,5.13
60,2026-04-06,5.13


### Reading T-Bills

In [4]:
def get_T_Bills(url):
  r = requests.get(url)

  r.raise_for_status()

  data2 = r.json()

  tbill_df = pd.DataFrame(data2)

  tbill_df = tbill_df['benchMarkTrendData'].apply(pd.Series)
  tbill_df = pd.concat([tbill_df.drop(columns= 'benchMarkTrend'),
                        tbill_df['benchMarkTrend'].apply(pd.Series)],
                      axis= 1)

  #flatten rates data for each date
  for i in range(61):
    tbill_df = pd.concat(
        [tbill_df.drop(columns= i),
        tbill_df[i].apply(pd.Series)],
        axis= 1
          )

  #stacks rate data of different dates into single column
  records2 = []

  for i in range(1, tbill_df.shape[1], 2):
    temp2 = pd.DataFrame({
        "productName": tbill_df.iloc[:, 0],
        "publishDate": tbill_df.iloc[:, i],
        "rate": tbill_df.iloc[:, i + 1]
    })

    records2.append(temp2)

  result2 = pd.concat(records2, ignore_index=True)

  result2["publishDate"] = pd.to_datetime(result2["publishDate"])

  final_tbill_df = result2.pivot(
      index= 'publishDate',
      columns= 'productName',
      values= 'rate'
  ).sort_index(ascending= False)

  return final_tbill_df


In [10]:
url = "https://www.fbil.org.in/wasdm/tbill/fetchcommongraph/3M"

tbill_df = get_T_Bills(url)

tbill_df.head()

productName,T-Bill Rate - 12 Months,T-Bill Rate - 3 Months,T-Bill Rate - 6 Months
publishDate,,,
2026-07-03,5.64,5.21,5.43
2026-07-02,5.64,5.24,5.44
2026-07-01,5.61,5.24,5.45
2026-06-30,5.50,5.23,5.30
2026-06-29,5.60,5.38,5.41


### Reading Term MIBOR

In [7]:
def get_Term_MIBOR(url):
  r3 = requests.get(url)

  r3.raise_for_status()

  data3 = r3.json()

  df = pd.DataFrame(data3)

  df = df['benchMarkTrendData'].apply(pd.Series)

  df = pd.concat([df.drop(columns= 'benchMarkTrend'),
                  df['benchMarkTrend'].apply(pd.Series)],
                axis= 1)

  for i in range(61):
    df = pd.concat([df.drop(columns= i),
                  df[i].apply(pd.Series)],
                axis= 1)

  #stacks rate data of different dates into single column
  records3 = []

  for i in range(1, df.shape[1], 2):
    temp3 = pd.DataFrame({
        "productName": df.iloc[:, 0],
        "publishDate": df.iloc[:, i],
        "rate": df.iloc[:, i + 1]
    })

    records3.append(temp3)

  result3 = pd.concat(records3, ignore_index=True)

  result3["publishDate"] = pd.to_datetime(result3["publishDate"])

  term_mibor_df = result3.pivot(
      index= 'publishDate',
      columns= 'productName',
      values= 'rate'
  ).sort_index(ascending= False)

  return term_mibor_df

In [9]:
term_mibor_url = "https://www.fbil.org.in/wasdm/termmibor/fetchcommongraph/3M"

term_mibor_df = get_Term_MIBOR(term_mibor_url)

term_mibor_df.head()

productName,Term MIBOR - 1 MONTH,Term MIBOR - 3 MONTHS
publishDate,,
2026-07-03,6.08,6.50
2026-07-02,6.10,6.52
2026-07-01,6.13,6.56
2026-06-30,6.20,6.61
2026-06-29,6.19,6.61
